In [ ]:
import os
from datetime import datetime

import pandas as pd

from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    isnan,
    lit,
    sum as spark_sum,
    when
)

In [ ]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
spark = (
    SparkSession.builder
    .appName("TechChallenge_DataQuality")
    .getOrCreate()
)

print("Spark iniciado.")

Spark iniciado.


In [ ]:
BASE_PATH = "/content/drive/MyDrive/TechChallenge_Fase2"

SILVER_PATH = f"{BASE_PATH}/data/silver_bigquery"
OUTPUT_PATH = f"{BASE_PATH}/outputs/data_quality"

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
TABELAS_SILVER = [
    "municipio",
    "uf",
    "meta_municipio",
    "meta_uf",
    "meta_brasil",
    "alunos_agregados"
]

silver = {
    nome: spark.read.parquet(
        f"{SILVER_PATH}/{nome}"
    )
    for nome in TABELAS_SILVER
}

for nome, df in silver.items():
    print(
        f"{nome}: {df.count():,} registros "
        f"e {len(df.columns)} colunas"
    )

municipio: 23,995 registros e 20 colunas
uf: 145 registros e 20 colunas
meta_municipio: 10,704 registros e 17 colunas
meta_uf: 81 registros e 16 colunas
meta_brasil: 3 registros e 15 colunas
alunos_agregados: 79,273 registros e 18 colunas


In [ ]:
def contar_nulos(df: DataFrame) -> dict:
    """
    Conta valores NULL por coluna.

    Em colunas numéricas double/float, também considera NaN.
    """

    resultado = {}

    tipos = dict(df.dtypes)

    for nome_coluna in df.columns:
        condicao = col(nome_coluna).isNull()

        if tipos[nome_coluna] in ["double", "float"]:
            condicao = (
                condicao
                | isnan(col(nome_coluna))
            )

        quantidade = (
            df.filter(condicao).count()
        )

        resultado[nome_coluna] = quantidade

    return resultado

In [ ]:
def analisar_tabela(
    nome: str,
    df: DataFrame
) -> dict:
    """
    Produz métricas gerais de qualidade.
    """

    total_registros = df.count()

    total_distintos = df.dropDuplicates().count()

    duplicados_completos = (
        total_registros - total_distintos
    )

    nulos_por_coluna = contar_nulos(df)

    total_nulos = sum(
        nulos_por_coluna.values()
    )

    return {
        "tabela": nome,
        "registros": total_registros,
        "colunas": len(df.columns),
        "duplicados_completos": duplicados_completos,
        "valores_nulos": total_nulos,
        "colunas_com_nulos": sum(
            1
            for valor in nulos_por_coluna.values()
            if valor > 0
        )
    }

In [ ]:
resultado_geral = []

for nome, df in silver.items():
    resultado_geral.append(
        analisar_tabela(nome, df)
    )

relatorio_geral = pd.DataFrame(
    resultado_geral
)

relatorio_geral

,tabela,registros,colunas,duplicados_completos,valores_nulos,colunas_com_nulos
0,municipio,23995,20,0,103923,9
1,uf,145,20,0,630,9
2,meta_municipio,10704,17,0,600,4
3,meta_uf,81,16,0,30,9
4,meta_brasil,3,15,0,0,0
5,alunos_agregados,79273,18,0,840,2


In [ ]:
resultados_testes = []


def registrar_teste(
    tabela: str,
    regra: str,
    quantidade_invalida: int,
    criticidade: str = "erro"
):
    status = (
        "APROVADO"
        if quantidade_invalida == 0
        else "REPROVADO"
    )

    resultados_testes.append({
        "tabela": tabela,
        "regra": regra,
        "registros_invalidos": int(
            quantidade_invalida
        ),
        "criticidade": criticidade,
        "status": status,
        "data_execucao": datetime.now()
    })

In [ ]:
municipio = silver["municipio"]

invalidos = municipio.filter(
    col("ano").isNull()
    | col("id_municipio").isNull()
    | col("rede").isNull()
).count()

registrar_teste(
    "municipio",
    "Chaves obrigatórias preenchidas",
    invalidos
)

In [ ]:
invalidos = municipio.filter(
    col("taxa_alfabetizacao").isNotNull()
    & (
        (col("taxa_alfabetizacao") < 0)
        | (col("taxa_alfabetizacao") > 100)
    )
).count()

registrar_teste(
    "municipio",
    "Taxa de alfabetização entre 0 e 100",
    invalidos
)

In [ ]:
duplicados_chave = (
    municipio
    .groupBy(
        "ano",
        "id_municipio",
        "serie",
        "rede"
    )
    .count()
    .filter(
        col("count") > 1
    )
    .count()
)

registrar_teste(
    "municipio",
    "Unicidade da chave ano-municipio-serie-rede",
    duplicados_chave
)

In [ ]:
uf = silver["uf"]

invalidos = uf.filter(
    col("ano").isNull()
    | col("sigla_uf").isNull()
    | col("rede").isNull()
).count()

registrar_teste(
    "uf",
    "Chaves obrigatórias preenchidas",
    invalidos
)

In [ ]:
invalidos = uf.filter(
    col("taxa_alfabetizacao").isNotNull()
    & (
        (col("taxa_alfabetizacao") < 0)
        | (col("taxa_alfabetizacao") > 100)
    )
).count()

registrar_teste(
    "uf",
    "Taxa de alfabetização entre 0 e 100",
    invalidos
)

In [ ]:
siglas_invalidas = uf.filter(
    col("sigla_uf").isNotNull()
    & (
        col("sigla_uf").rlike(
            "^[A-Z]{2}$"
        ) == False
    )
).count()

registrar_teste(
    "uf",
    "Sigla da UF com duas letras maiúsculas",
    siglas_invalidas
)

In [ ]:
COLUNAS_META = [
    f"meta_alfabetizacao_{ano}"
    for ano in range(2024, 2031)
]


def validar_tabela_meta(
    nome: str,
    df: DataFrame
):
    colunas_disponiveis = [
        coluna
        for coluna in COLUNAS_META
        if coluna in df.columns
    ]

    for nome_coluna in colunas_disponiveis:
        invalidos = df.filter(
            col(nome_coluna).isNotNull()
            & (
                (col(nome_coluna) < 0)
                | (col(nome_coluna) > 100)
            )
        ).count()

        registrar_teste(
            nome,
            f"{nome_coluna} entre 0 e 100",
            invalidos
        )

    if "percentual_participacao" in df.columns:
        invalidos = df.filter(
            col("percentual_participacao").isNotNull()
            & (
                (col("percentual_participacao") < 0)
                | (col("percentual_participacao") > 100)
            )
        ).count()

        registrar_teste(
            nome,
            "Percentual de participação entre 0 e 100",
            invalidos
        )

In [ ]:
validar_tabela_meta(
    "meta_municipio",
    silver["meta_municipio"]
)

validar_tabela_meta(
    "meta_uf",
    silver["meta_uf"]
)

validar_tabela_meta(
    "meta_brasil",
    silver["meta_brasil"]
)

In [ ]:
alunos = silver["alunos_agregados"]

invalidos = alunos.filter(
    col("ano").isNull()
    | col("id_municipio").isNull()
    | col("total_registros").isNull()
).count()

registrar_teste(
    "alunos_agregados",
    "Chaves e total de registros preenchidos",
    invalidos
)

In [ ]:
colunas_totais = [
    "total_registros",
    "total_presentes",
    "total_alfabetizados",
    "total_cadernos_preenchidos"
]

for nome_coluna in colunas_totais:
    invalidos = alunos.filter(
        col(nome_coluna).isNotNull()
        & (col(nome_coluna) < 0)
    ).count()

    registrar_teste(
        "alunos_agregados",
        f"{nome_coluna} não negativo",
        invalidos
    )

In [ ]:
invalidos = alunos.filter(
    col("total_presentes")
    > col("total_registros")
).count()

registrar_teste(
    "alunos_agregados",
    "Presentes não excedem total de registros",
    invalidos
)

In [ ]:
invalidos = alunos.filter(
    col("total_alfabetizados")
    > col("total_registros")
).count()

registrar_teste(
    "alunos_agregados",
    "Alfabetizados não excedem total de registros",
    invalidos
)

In [ ]:
invalidos = alunos.filter(
    col("total_cadernos_preenchidos")
    > col("total_registros")
).count()

registrar_teste(
    "alunos_agregados",
    "Cadernos preenchidos não excedem total de registros",
    invalidos
)

In [ ]:
colunas_taxa = [
    "taxa_presenca",
    "taxa_alfabetizacao_alunos",
    "taxa_preenchimento_caderno"
]

for nome_coluna in colunas_taxa:
    invalidos = alunos.filter(
        col(nome_coluna).isNotNull()
        & (
            (col(nome_coluna) < 0)
            | (col(nome_coluna) > 100)
        )
    ).count()

    registrar_teste(
        "alunos_agregados",
        f"{nome_coluna} entre 0 e 100",
        invalidos
    )

In [ ]:
invalidos = alunos.filter(
    col("proficiencia_media").isNotNull()
    & (col("proficiencia_media") < 0)
).count()

registrar_teste(
    "alunos_agregados",
    "Proficiência média não negativa",
    invalidos
)

In [ ]:
duplicados_alunos = (
    alunos
    .groupBy(
        "ano",
        "id_municipio",
        "id_escola",
        "serie",
        "rede"
    )
    .count()
    .filter(
        col("count") > 1
    )
    .count()
)

registrar_teste(
    "alunos_agregados",
    (
        "Unicidade da chave "
        "ano-municipio-escola-serie-rede"
    ),
    duplicados_alunos
)

In [ ]:
municipios_referencia = (
    municipio
    .select("id_municipio")
    .distinct()
)

municipios_alunos = (
    alunos
    .select("id_municipio")
    .distinct()
)

In [ ]:
municipios_sem_referencia = (
    municipios_alunos.alias("a")
    .join(
        municipios_referencia.alias("m"),
        on="id_municipio",
        how="left_anti"
    )
    .count()
)

registrar_teste(
    "alunos_agregados",
    "Municípios existentes na tabela município",
    municipios_sem_referencia,
    criticidade="alerta"
)

In [ ]:
relatorio_testes = pd.DataFrame(
    resultados_testes
)

relatorio_testes

,tabela,regra,registros_invalidos,criticidade,status,data_execucao
0,municipio,Chaves obrigatórias preenchidas,0,erro,APROVADO,2026-07-14 04:44:10.466201
1,municipio,Taxa de alfabetização entre 0 e 100,0,erro,APROVADO,2026-07-14 04:44:10.798579
2,municipio,Unicidade da chave ano-municipio-serie-rede,0,erro,APROVADO,2026-07-14 04:44:11.576588
3,uf,Chaves obrigatórias preenchidas,0,erro,APROVADO,2026-07-14 04:44:11.783247
4,uf,Taxa de alfabetização entre 0 e 100,0,erro,APROVADO,2026-07-14 04:44:12.020260
5,uf,Sigla da UF com duas letras maiúsculas,0,erro,APROVADO,2026-07-14 04:44:12.473590
6,meta_municipio,meta_alfabetizacao_2024 entre 0 e 100,0,erro,APROVADO,2026-07-14 04:44:12.691964
7,meta_municipio,meta_alfabetizacao_2025 entre 0 e 100,0,erro,APROVADO,2026-07-14 04:44:12.885594
8,meta_municipio,meta_alfabetizacao_2026 entre 0 e 100,0,erro,APROVADO,2026-07-14 04:44:13.114535
9,meta_municipio,meta_alfabetizacao_2027 entre 0 e 100,0,erro,APROVADO,2026-07-14 04:44:13.315760


In [ ]:
resumo_testes = (
    relatorio_testes
    .groupby(
        ["status", "criticidade"]
    )
    .size()
    .reset_index(
        name="quantidade"
    )
)

resumo_testes

,status,criticidade,quantidade
0,APROVADO,erro,43
1,REPROVADO,alerta,1


In [ ]:
erros_reprovados = relatorio_testes[
    (relatorio_testes["status"] == "REPROVADO")
    & (relatorio_testes["criticidade"] == "erro")
]

if erros_reprovados.empty:
    STATUS_PIPELINE = "APROVADA"
    print("Camada Silver aprovada nos testes críticos.")
else:
    STATUS_PIPELINE = "REPROVADA"

    print("Existem testes críticos reprovados.")

    display(
        erros_reprovados
    )

Camada Silver aprovada nos testes críticos.


In [ ]:
relatorio_geral.to_csv(
    f"{OUTPUT_PATH}/resumo_qualidade_tabelas.csv",
    index=False
)

relatorio_testes.to_csv(
    f"{OUTPUT_PATH}/resultado_regras_qualidade.csv",
    index=False
)

resumo_testes.to_csv(
    f"{OUTPUT_PATH}/resumo_status_qualidade.csv",
    index=False
)

print(
    f"Relatórios salvos em: {OUTPUT_PATH}"
)

Relatórios salvos em: /content/drive/MyDrive/TechChallenge_Fase2/outputs/data_quality


In [ ]:
if STATUS_PIPELINE == "REPROVADA":
    raise RuntimeError(
        "Camada Silver reprovada nos testes de qualidade."
    )

print(
    "Dados liberados para processamento na Gold."
)

Dados liberados para processamento na Gold.
